In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW sales_raw_view AS
SELECT * FROM (
    SELECT distinct sale_id, customer_id, product_id, quantity,
           coalesce(sale_amount, 0) as sale_amount, sale_date, region, ingestion_date
    FROM identifier(:catalog || '.bronze.sales_raw')
    WHERE sale_id IS NOT NULL
)
QUALIFY row_number() OVER (PARTITION BY sale_id ORDER BY ingestion_date DESC) = 1

In [0]:
%sql
CREATE TABLE IF NOT EXISTS identifier(:catalog || '.silver.sales_clean') (
    sale_id STRING,
    customer_id INT,
    product_id INT,
    quantity INT,
    sale_amount DECIMAL(10,2),
    sale_date DATE,
    region STRING
)

In [0]:
%sql
MERGE INTO identifier(:catalog || '.silver.sales_clean') t
USING sales_raw_view s
ON t.sale_id = s.sale_id
WHEN MATCHED THEN UPDATE SET
    t.customer_id = s.customer_id, t.product_id = s.product_id, t.quantity = s.quantity,
    t.sale_amount = s.sale_amount, t.sale_date = s.sale_date, t.region = s.region
WHEN NOT MATCHED THEN INSERT (sale_id, customer_id, product_id, quantity, sale_amount, sale_date, region)
VALUES (s.sale_id, s.customer_id, s.product_id, s.quantity, s.sale_amount, s.sale_date, s.region);
 